In [12]:
import requests
import pandas as pd

## Получим визиты:

In [13]:
url_visits = "https://data-charts-api.hexlet.app/visits?begin=2023-03-01&end=2023-09-01"

response = requests.get(url_visits)

visits = pd.DataFrame(response.json())

## Получим регистрации:

In [14]:
url_registrations = "https://data-charts-api.hexlet.app/registrations?begin=2023-03-01&end=2023-09-01"

response = requests.get(url_registrations)

registrations = pd.DataFrame(response.json())

## Убрать ботов:

In [34]:
visits = visits[~visits['user_agent'].str.lower().str.contains('bot')]

## Берём только последний визит пользователя

Преобразуем дату:

In [15]:
visits['datetime'] = pd.to_datetime(visits['datetime'])

Отсортируем:

In [16]:
visits = visits.sort_values('datetime')

Оставляем последний визит:

In [17]:
visits = visits.drop_duplicates(
    subset='visit_id',
    keep='last'
)

## Получим дату для группировки

In [18]:
visits['date_group'] = visits['datetime'].dt.date

In [19]:
registrations['datetime'] = pd.to_datetime(registrations['datetime'])

registrations['date_group'] = registrations['datetime'].dt.date

## Группируем визиты

In [44]:
visits_grouped = (
    visits
    .groupby(['date_group', 'platform'])
    .agg(visits=('visit_id', 'count'))
    .reset_index()
)

## Группируем регистрации

In [21]:
registrations_grouped = (
    registrations
    .groupby(['date_group', 'platform'])
    .agg(registrations=('user_id', 'count'))
    .reset_index()
)

## Объединяем таблицы

In [22]:
conversion = visits_grouped.merge(
    registrations_grouped,
    on=['date_group', 'platform'],
    how='left'
)

## Заполняем пропуски

In [23]:
conversion['registrations'] = conversion['registrations'].fillna(0)

## Считаем конверсию

In [24]:
conversion['conversion'] = (
    conversion['registrations'] /
    conversion['visits']
)

## Сортируем по дате

In [45]:
conversion = conversion.sort_values('date_group')

## Сохраняем JSON

In [28]:
conversion.to_json('./conversion.json')